In [ ]:
# =====================================================================
# LABEL RELIABILITY AUDIT -- TruthfulQA MC1 (simple version). Paste each CELL into its own Colab cell.
#   CELL 0: mount Drive, define RESULTS_DIR
#   CELL 1: run ONCE (it refuses to run twice). Draws the 60 items, writes the blind sheets + the key.
#   CELL 1b: run ONCE right after CELL 1. Converts the sheets to Excel files with dropdowns -> annotators work on the .xlsx files
#   CELL 2: run AFTER both annotators finish. Unblinds, writes the disagreement sheets (Excel, typology dropdown) to code.
#   CELL 3: run AFTER typology is coded (same session as CELL 2; after a reset run 0, 2, 3 -- never 1 or 1b).
#           Disagreement rate, typology counts, Cohen's kappa.
# =====================================================================


In [6]:
# %% ===== CELL 0: setup =====
import os, glob
from google.colab import drive
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/thesis_results"
assert glob.glob(os.path.join(RESULTS_DIR, "exports", "truthfulqa", "per_item_gemma-3-12b_*.csv")), "per-item export not found"

import re, pandas as pd
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Alignment, PatternFill, Font
from openpyxl.utils import get_column_letter

CODES = ["INCORRECT_LABEL", "AMBIGUOUS_WORDING", "CONTESTED_OR_EVOLVING", "INSUFFICIENT_EVIDENCE", "INTERPRETATION_DISAGREEMENT"]

def write_xlsx(df, path, dropdowns, input_cols):
    """Excel file with frozen header, wrapped text, yellow input columns and dropdowns.
    dropdowns: {column: list of allowed values, or function(row_number)->list}."""
    widths = {"item_id": 9, "category": 16, "question": 45, "pick": 14, "evidence_url": 32, "notes": 32, "typology_code": 30, "rationale": 45}
    with pd.ExcelWriter(path, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name="audit"); ws = w.book[w.book.sheetnames[0]]
        ws.freeze_panes = "A2"; yellow = PatternFill("solid", fgColor="FFF2CC")
        for j, c in enumerate(df.columns, 1):
            L = get_column_letter(j); ws.column_dimensions[L].width = widths.get(c, 30)
            ws.cell(1, j).font = Font(bold=True)
            if c in input_cols: ws.cell(1, j).fill = yellow
            for r in range(2, len(df) + 2):
                ws.cell(r, j).alignment = Alignment(wrap_text=True, vertical="top")
                if c in input_cols: ws.cell(r, j).fill = yellow
            if c in dropdowns:
                spec = dropdowns[c]
                for r in range(2, len(df) + 2):
                    vals = spec(r - 2) if callable(spec) else spec
                    dv = DataValidation(type="list", formula1='"' + ",".join(vals) + '"', allow_blank=True, showErrorMessage=True,
                                        errorTitle="Not allowed", error="Choose a value from the dropdown list.")
                    ws.add_data_validation(dv); dv.add(f"{L}{r}")

def rd_x(path):
    d = pd.read_excel(path, dtype=str).fillna("")
    for c in d.columns: d[c] = d[c].astype(str).str.strip()
    if "pick" in d.columns: d["pick"] = d["pick"].str.replace(r"\.0$", "", regex=True)   # Excel may store 3 as 3.0
    return d




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# %% ===== CELL 1: draw sample + blind sheets (run once) =====
import os, re, glob, json, random, hashlib
import numpy as np, pandas as pd
from datasets import load_dataset

SEED, N_AUDIT, N_SECOND = 42, 60, 20
AUDIT_DIR = os.path.join(RESULTS_DIR, "label_audit"); os.makedirs(AUDIT_DIR, exist_ok=True)
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().casefold()

def allocate(counts, n):                       # proportional, largest-remainder
    tot = sum(counts.values()); q = {c: n * k / tot for c, k in counts.items()}
    a = {c: int(np.floor(v)) for c, v in q.items()}
    for c in sorted(counts, key=lambda c: (-(q[c] - a[c]), c))[: n - sum(a.values())]: a[c] += 1
    return a

# the 300 evaluated questions, and the dataset version that was evaluated
ev = pd.read_csv(sorted(glob.glob(os.path.join(RESULTS_DIR, "exports", "truthfulqa", "per_item_gemma-3-12b_*.csv")))[-1])[["question", "category"]]
assert len(ev) == 300
gen = {norm(r["question"]): r for r in load_dataset("truthfulqa/truthful_qa", "generation")["validation"]}
mc  = {norm(r["question"]): r for r in load_dataset("truthfulqa/truthful_qa", "multiple_choice")["validation"]}
FIX = {norm("What happens if you turn over a tarot card showing Death?"): "Paranormal"}   # category recorded as "unknown" by the evaluation loader
items = {}
for q, c in zip(ev["question"], ev["category"]):
    k = norm(q); assert k in mc, f"not found in dataset: {q!r}"
    cat = gen[k]["category"] if k in gen else FIX.get(k, c)
    assert cat != "unknown", f"unresolved category: {q!r}"
    items[k] = {"question": q, "category": cat, "choices": list(mc[k]["mc1_targets"]["choices"]),
                "labels": list(mc[k]["mc1_targets"]["labels"]), "source": gen[k]["source"] if k in gen else ""}

# draw (deterministic)
by_cat = {}
for k, it in items.items(): by_cat.setdefault(it["category"], []).append(k)
alloc = allocate({c: len(v) for c, v in by_cat.items()}, N_AUDIT)
rng = random.Random(SEED); audit = []
for c in sorted(by_cat): audit += rng.sample(sorted(by_cat[c]), alloc[c]) if alloc[c] else []
second = set(random.Random(SEED + 2).sample(sorted(audit), N_SECOND))
sample_hash = hashlib.sha256("\n".join(sorted(audit)).encode()).hexdigest()

# blind sheets: item order and option order shuffled, so position never reveals the label
rng2 = random.Random(SEED + 3); order = list(audit); rng2.shuffle(order)
blind, key = [], []
for i, k in enumerate(order, 1):
    it = items[k]; assert sum(it["labels"]) == 1, "MC1 item without exactly one correct option"
    opts = list(zip(it["choices"], it["labels"])); rng2.shuffle(opts)
    row = {"item_id": f"A{i:03d}", "category": it["category"], "question": it["question"]}
    for j, (ch, _) in enumerate(opts, 1): row[f"option_{j}"] = ch
    blind.append(row)
    key.append({"item_id": f"A{i:03d}", "dataset_correct_option": [lb for _, lb in opts].index(1) + 1,
                "second_annotator": k in second, "authors_source": it["source"]})
assert not any(os.path.exists(os.path.join(AUDIT_DIR, f"sheet_annotator1_ALL60.{e}")) for e in ("csv", "xlsx")), \
    "Sheets already exist -- CELL 1 must run only once (re-running would erase filled-in work). Do not re-run it."
blind = pd.DataFrame(blind); key = pd.DataFrame(key)
opt_cols = [c for c in blind.columns if c.startswith("option_")]
for c in ("pick", "evidence_url", "notes"): blind[c] = ""          # pick = option number, or NONE, or CANNOT_DETERMINE
blind = blind[["item_id", "category", "question"] + opt_cols + ["pick", "evidence_url", "notes"]]
sub = blind[blind["item_id"].isin(key.loc[key["second_annotator"], "item_id"])]
blind.to_csv(os.path.join(AUDIT_DIR, "sheet_annotator1_ALL60.csv"), index=False, encoding="utf-8-sig")
sub.to_csv(os.path.join(AUDIT_DIR, "sheet_annotator2_SUBSET20.csv"), index=False, encoding="utf-8-sig")
key.to_csv(os.path.join(AUDIT_DIR, "KEY_do_not_open_until_both_annotators_finish.csv"), index=False)
print(f"{len(blind)} items, {len(sub)} for annotator 2.  Sample SHA-256 (record in your protocol): {sample_hash}")
print("Give annotators ONLY the two sheet_ files. Do not open the KEY file.")


60 items, 20 for annotator 2.  Sample SHA-256 (record in your protocol): a45cb11f70ad10e795719c5b3adffb5db26fc61c9bca68b47c9aab1ba1206bbc
Give annotators ONLY the two sheet_ files. Do not open the KEY file.


In [ ]:
# %% ===== CELL 1b: convert the sheets to Excel with dropdowns (run once, right after CELL 1) =====
import os, shutil
AUDIT_DIR = os.path.join(RESULTS_DIR, "label_audit")
OLD = os.path.join(AUDIT_DIR, "_original_csv"); os.makedirs(OLD, exist_ok=True)
for name in ("sheet_annotator1_ALL60", "sheet_annotator2_SUBSET20"):
    src = os.path.join(AUDIT_DIR, name + ".csv")
    if not os.path.exists(src): print(name + ".csv not found (already converted?)"); continue
    df = pd.read_csv(src, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    df = df.loc[:, ~(df.columns.str.startswith("option_") & df.eq("").all())]                     # drop option columns that are empty
    n_opts = df[[c for c in df.columns if c.startswith("option_")]].ne("").sum(axis=1)
    write_xlsx(df, os.path.join(AUDIT_DIR, name + ".xlsx"),
               {"pick": lambda i: [str(k) for k in range(1, n_opts[i] + 1)] + ["NONE", "CANNOT_DETERMINE"]}, ["pick", "evidence_url", "notes"])
    shutil.move(src, os.path.join(OLD, name + ".csv"))
    print("created", name + ".xlsx  (give this one to the annotator)")




created sheet_annotator1_ALL60.xlsx  (give this one to the annotator)
created sheet_annotator2_SUBSET20.xlsx  (give this one to the annotator)


In [11]:
# %% ===== CELL 2: unblind (run after both annotators finish) =====
import os, glob
import pandas as pd
AUDIT_DIR = os.path.join(RESULTS_DIR, "label_audit")
rd = lambda p: pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig")   # for the key
key = rd(os.path.join(AUDIT_DIR, "KEY_do_not_open_until_both_annotators_finish.csv"))

def unblind(sheet_path, out_path):
    s = rd_x(sheet_path); s["pick"] = s["pick"].str.upper()
    n_opts = s[[c for c in s.columns if c.startswith("option_")]].ne("").sum(axis=1)
    bad = s[~s.apply(lambda r: r["pick"] in {"NONE", "CANNOT_DETERMINE"} or (r["pick"].isdigit() and 1 <= int(r["pick"]) <= n_opts[r.name]), axis=1)]
    assert bad.empty, f"{len(bad)} rows have a blank/invalid pick (use an option number, NONE or CANNOT_DETERMINE): {bad['item_id'].tolist()[:8]}"
    m = s.merge(key, on="item_id", validate="one_to_one")
    m["agrees_with_dataset"] = (m["pick"] == m["dataset_correct_option"]).astype(int)
    m["dataset_answer"] = m.apply(lambda r: r[f"option_{r['dataset_correct_option']}"], axis=1)
    t = m.loc[m["agrees_with_dataset"] == 0, ["item_id", "question", "dataset_answer", "pick", "evidence_url", "notes", "authors_source"]].copy()
    t["typology_code"] = ""; t["rationale"] = ""
    if os.path.exists(out_path): print(f"{os.path.basename(out_path)} already exists -- left untouched")
    else:
        write_xlsx(t, out_path, {"typology_code": CODES}, ["typology_code", "rationale"])
        print(f"{os.path.basename(sheet_path)}: {len(m)} items, {len(t)} disagree with the dataset -> {os.path.basename(out_path)}")
    return m

m1 = unblind(os.path.join(AUDIT_DIR, "sheet_annotator1_ALL60.xlsx"), os.path.join(AUDIT_DIR, "TO_CODE_annotator1.xlsx"))
m2 = unblind(os.path.join(AUDIT_DIR, "sheet_annotator2_SUBSET20.xlsx"), os.path.join(AUDIT_DIR, "TO_CODE_annotator2.xlsx"))
print("\nEach annotator now picks a typology_code from the dropdown and writes a one-line rationale in their own TO_CODE .xlsx file.")


TO_CODE_annotator1.xlsx already exists -- left untouched
TO_CODE_annotator2.xlsx already exists -- left untouched

Each annotator now picks a typology_code from the dropdown and writes a one-line rationale in their own TO_CODE .xlsx file.


In [12]:
# %% ===== CELL 3: report (run after both TO_CODE files are filled) =====
import numpy as np
from sklearn.metrics import cohen_kappa_score
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z**2 / n; c = (p + z**2 / (2*n)) / d
    h = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / d
    return c - h, c + h

def coded(path):
    t = rd_x(path)
    assert t["typology_code"].isin(CODES).all(), f"{path}: blank/invalid typology_code on {t.loc[~t['typology_code'].isin(CODES), 'item_id'].tolist()}"
    return t

t1 = coded(os.path.join(AUDIT_DIR, "TO_CODE_annotator1.xlsx")); t2 = coded(os.path.join(AUDIT_DIR, "TO_CODE_annotator2.xlsx"))
n, d = len(m1), int((m1["agrees_with_dataset"] == 0).sum()); lo, hi = wilson(d, n)
print(f"Annotator 1: {d}/{n} items disagree with the dataset = {d/n:.1%}  (95% Wilson CI {lo:.1%}-{hi:.1%});  paper's validator disagreement: ~7%")
print("\nTypology (annotator 1):"); print(t1["typology_code"].value_counts().reindex(CODES, fill_value=0).to_string())

a = m1[m1["item_id"].isin(m2["item_id"])].set_index("item_id")["agrees_with_dataset"].sort_index()
b = m2.set_index("item_id")["agrees_with_dataset"].sort_index()
kappa = cohen_kappa_score(a, b) if len(set(a) | set(b)) > 1 else float("nan")
print(f"\nSecond annotator ({len(a)} items): agree/disagree-with-dataset, percent agreement = {(a == b).mean():.1%}, Cohen's kappa = {kappa:.3f}")
both = sorted(set(t1["item_id"]) & set(t2["item_id"]))
if both:
    c1 = t1.set_index("item_id").loc[both, "typology_code"]; c2 = t2.set_index("item_id").loc[both, "typology_code"]
    print(f"Typology agreement on the {len(both)} items both flagged: {(c1 == c2).mean():.1%}")

Annotator 1: 12/60 items disagree with the dataset = 20.0%  (95% Wilson CI 11.8%-31.8%);  paper's validator disagreement: ~7%

Typology (annotator 1):
typology_code
INCORRECT_LABEL                3
AMBIGUOUS_WORDING              0
CONTESTED_OR_EVOLVING          5
INSUFFICIENT_EVIDENCE          0
INTERPRETATION_DISAGREEMENT    4

Second annotator (20 items): agree/disagree-with-dataset, percent agreement = 80.0%, Cohen's kappa = 0.467
Typology agreement on the 3 items both flagged: 66.7%
